### Import libraries

In [2]:
import pandas as pd
import numpy as np
import json
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

### Data Loading

In [3]:
data = pd.read_csv("/home/ferrara/genai/data/genes_subtypes_brca_cleaned")
columns = list(data.columns)

In [5]:
print("Data shape is:", data.shape)
print("Overview of data:\n", data.head())

Data shape is: (1168, 18473)
Overview of data:
      Unnamed: 0  FAM149B1      NOX4    POU6F1     FBXW7     SELPLG       LDB1  \
0  TCGA-3C-AAAU  2.922541  1.021608  1.882429  1.400295   5.907222  32.313730   
1  TCGA-3C-AALI  2.692196  1.327865  1.509737  2.480337  13.738894  18.290485   
2  TCGA-3C-AALJ  5.325364  2.615592  1.060712  2.141253  13.849035  17.378361   
3  TCGA-3C-AALK  3.520777  2.013150  0.975878  1.635825  10.191759  33.762770   
4  TCGA-4H-AAAK  4.210197  3.856625  2.144672  1.779310  11.736018  34.514200   

    METTL15       TBCD      NUDT9  ...      TRIL    HSD3B1  LCE3B       ABI3  \
0  3.290892  10.594536   7.688316  ...  5.491752  0.000000    0.0   3.399687   
1  1.681634  16.985263   7.182599  ...  8.588787  0.000000    0.0  10.175611   
2  2.694136  12.502003  13.740586  ...  5.995271  0.000000    0.0   4.705460   
3  1.809036  10.889388  14.167553  ...  7.441441  0.091334    0.0   4.557695   
4  2.471262   8.889673  10.505968  ...  7.600688  0.000000    0.0

In [6]:
with open("/home/ferrara/genai/data/BRCA1_DN.V1_DN.v2025.1.Hs.json", mode='r') as BRCA1_DN_file:
    BRCA1_DN = json.load(BRCA1_DN_file)

In [7]:
with open("/home/ferrara/genai/data/BMI1_DN.V1_UP.v2025.1.Hs.json", mode='r') as BRCA1_UP_file:
    BRCA1_UP = json.load(BRCA1_UP_file)

In [8]:
BRCA1_DN_genes = BRCA1_DN["BRCA1_DN.V1_DN"]["geneSymbols"]
BRCA1_UP_genes = BRCA1_UP["BMI1_DN.V1_UP"]["geneSymbols"]

In [22]:
BRCA_UP_DN = np.unique(BRCA1_DN_genes + BRCA1_UP_genes)
BRCA_UP_DN_in_data = [x for x in BRCA_UP_DN if x in columns]
filter_cols = BRCA_UP_DN_in_data + ["Expert subtype"]

In [23]:
genes_up_dn = pd.DataFrame(BRCA_UP_DN_in_data)
genes_up_dn.to_csv("/home/ferrara/genai/data/genes.csv", index=False)

### Data preprocessing

In [24]:
X = data[filter_cols]

In [25]:
y = data["Expert subtype"]

In [26]:
print("Data shape is:", X.shape)
print("Overview of data:\n", X.head())

Data shape is: (1168, 264)
Overview of data:
       ABCA1      ACKR3       ACTA2     ADAM12     ADRB2       AGRN  AMBN  \
0  1.433302   6.707991   53.583332   5.590052  0.250527  38.875565   0.0   
1  2.014925  16.300197   97.031140   8.836612  0.251083  13.404982   0.0   
2  3.379976   8.735359   89.748012  11.033321  0.492126  44.661484   0.0   
3  3.181489   9.321796  174.178411  21.447428  0.658870  56.365021   0.0   
4  3.418564  20.043065  129.854572  26.363304  0.726293  47.757600   0.0   

        ANPEP     ANXA3     APOA4  ...      VNN1     WFDC1      XCL1  \
0    0.541870  0.370416  0.011712  ...  0.166626  0.149243  0.037859   
1    2.355572  0.925539  0.000000  ...  0.345798  0.152079  0.440758   
2   19.586780  0.407067  0.000000  ...  0.657424  0.182982  0.730984   
3   19.132425  8.308039  0.000000  ...  0.341233  0.579944  0.421668   
4  166.790628  9.254230  0.000000  ...  0.798496  0.179654  0.269894   

     ZBTB44   ZMYND10    ZNF124    ZNF253      ZNF32    ZNF668  

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify = y)

print("Training Data Shape:", X_train.shape)
print("Training Label Shape:", y_train.shape)
print("Validation Data Shape:", X_test.shape)
print("Validation Label Shape:", y_test.shape)

Training Data Shape: (1051, 264)
Training Label Shape: (1051,)
Validation Data Shape: (117, 264)
Validation Label Shape: (117,)


### Models

In [28]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dims=(128, 128), dropout=0.0, final_act=None):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(d, h),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, out_dim))
        if final_act is not None:
            layers.append(final_act)
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class Generator(nn.Module):
    def __init__(self, z_dim, data_dim, hidden_dims=(128, 128)):
        super().__init__()
        # No final activation because we're generating standardized (roughly Gaussian-ish) features.
        self.net = MLP(z_dim, data_dim, hidden_dims=hidden_dims, dropout=0.0, final_act=None)

    def forward(self, z):
        return self.net(z)


class Critic(nn.Module):
    def __init__(self, data_dim, hidden_dims=(128, 128)):
        super().__init__()
        # Output is a scalar "critic score" (not a probability).
        self.net = MLP(data_dim, 1, hidden_dims=hidden_dims, dropout=0.0, final_act=None)

    def forward(self, x):
        return self.net(x).view(-1)

### WGAN-GP utilities

In [29]:
def gradient_penalty(critic, real, fake, device, gp_lambda=10.0):
    """
    Computes gradient penalty: E[(||∇_x_hat D(x_hat)||2 - 1)^2]
    where x_hat is interpolated between real and fake.
    """
    bsz = real.size(0)
    eps = torch.rand(bsz, 1, device=device)
    eps = eps.expand_as(real)

    x_hat = eps * real + (1 - eps) * fake
    x_hat.requires_grad_(True)

    d_hat = critic(x_hat)
    grads = torch.autograd.grad(
        outputs=d_hat,
        inputs=x_hat,
        grad_outputs=torch.ones_like(d_hat),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    grads = grads.view(bsz, -1)
    grad_norm = grads.norm(2, dim=1)
    gp = ((grad_norm - 1.0) ** 2).mean() * gp_lambda
    return gp


@dataclass
class WGANConfig:
    z_dim: int = 64
    g_hidden: tuple = (128, 128)
    d_hidden: tuple = (128, 128)
    batch_size: int = 64
    epochs: int = 200
    lr: float = 1e-4
    betas: tuple = (0.0, 0.9)     # common for WGAN-GP
    n_critic: int = 5            # critic updates per generator update
    gp_lambda: float = 10.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42


class TabularWGAN_GP:
    def __init__(self, data_dim: int, cfg: WGANConfig = WGANConfig()):
        self.data_dim = data_dim
        self.cfg = cfg

        torch.manual_seed(cfg.seed)
        np.random.seed(cfg.seed)

        self.device = torch.device(cfg.device)

        self.G = Generator(cfg.z_dim, data_dim, hidden_dims=cfg.g_hidden).to(self.device)
        self.D = Critic(data_dim, hidden_dims=cfg.d_hidden).to(self.device)

        self.opt_G = optim.Adam(self.G.parameters(), lr=cfg.lr, betas=cfg.betas)
        self.opt_D = optim.Adam(self.D.parameters(), lr=cfg.lr, betas=cfg.betas)

        self.scaler = StandardScaler()

    def fit(self, X: np.ndarray, verbose_every: int = 10):
        """
        X: numpy array shape (N, D), continuous-only.
        """
        assert X.ndim == 2 and X.shape[1] == self.data_dim, "X must be (N, data_dim)."

        # Scale to stabilize GAN training
        Xs = self.scaler.fit_transform(X).astype(np.float32)

        ds = TensorDataset(torch.from_numpy(Xs))
        dl = DataLoader(ds, batch_size=self.cfg.batch_size, shuffle=True, drop_last=True)

        self.G.train()
        self.D.train()

        steps = 0
        for epoch in range(1, self.cfg.epochs + 1):
            for (real_batch,) in dl:
                real = real_batch.to(self.device)
                bsz = real.size(0)

                # -------------------------
                # Train Critic (n_critic times)
                # -------------------------
                for _ in range(self.cfg.n_critic):
                    z = torch.randn(bsz, self.cfg.z_dim, device=self.device)
                    fake = self.G(z).detach()

                    d_real = self.D(real).mean()
                    d_fake = self.D(fake).mean()

                    gp = gradient_penalty(self.D, real, fake, self.device, gp_lambda=self.cfg.gp_lambda)

                    # WGAN objective: maximize D(real)-D(fake)
                    # We minimize negative of that + GP
                    d_loss = -(d_real - d_fake) + gp

                    self.opt_D.zero_grad(set_to_none=True)
                    d_loss.backward()
                    self.opt_D.step()

                # -------------------------
                # Train Generator (1 time)
                # -------------------------
                z = torch.randn(bsz, self.cfg.z_dim, device=self.device)
                fake = self.G(z)
                g_loss = -self.D(fake).mean()  # maximize D(fake)

                self.opt_G.zero_grad(set_to_none=True)
                g_loss.backward()
                self.opt_G.step()

                steps += 1

            if epoch % verbose_every == 0 or epoch == 1 or epoch == self.cfg.epochs:
                # Quick diagnostics on last batch
                with torch.no_grad():
                    wasserstein_est = (d_real - d_fake).item()
                print(
                    f"Epoch {epoch:4d}/{self.cfg.epochs} | "
                    f"D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f} | "
                    f"W_est: {wasserstein_est:.4f}"
                )

    @torch.no_grad()
    def sample(self, n: int) -> np.ndarray:
        """
        Returns synthetic samples in the ORIGINAL feature scale (inverse-transformed).
        """
        self.G.eval()
        z = torch.randn(n, self.cfg.z_dim, device=self.device)
        xs = self.G(z).cpu().numpy()
        x = self.scaler.inverse_transform(xs)
        return x

    def save(self, path: str):
        payload = {
            "data_dim": self.data_dim,
            "cfg": self.cfg.__dict__,
            "G": self.G.state_dict(),
            "D": self.D.state_dict(),
            "scaler_mean": self.scaler.mean_,
            "scaler_scale": self.scaler.scale_,
        }
        torch.save(payload, path)

    @staticmethod
    def load(path: str):
        payload = torch.load(path, map_location="cpu")
        cfg = WGANConfig(**payload["cfg"])
        model = TabularWGAN_GP(payload["data_dim"], cfg)
        model.G.load_state_dict(payload["G"])
        model.D.load_state_dict(payload["D"])
        model.scaler.mean_ = payload["scaler_mean"]
        model.scaler.scale_ = payload["scaler_scale"]
        model.scaler.var_ = model.scaler.scale_ ** 2
        model.scaler.n_features_in_ = payload["data_dim"]
        return model

### Training

In [34]:
N = X.shape[0]
D = X.shape[1]
X = np.random.randn(N, D) * np.linspace(1, 5, D) + np.linspace(-3, 3, D)

cfg = WGANConfig(
    z_dim=64,
    epochs=200,
    batch_size=64,
    n_critic=5,
    lr=1e-4,
    gp_lambda=10.0,
)

gan = TabularWGAN_GP(data_dim=D, cfg=cfg)
gan.fit(X, verbose_every=1)

X_syn = gan.sample(2000)
print("Synthetic shape:", X_syn.shape)


/home/ferrara/anaconda3/envs/gan/lib/python3.11/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/ferrara/anaconda3/envs/gan/lib/python3.11/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/ferrara/anaconda3/envs/gan/lib/python3.11/site-packages/torch/cuda/__init__.py:326: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instr

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [39]:
print("cuda:", torch.version.cuda)
print("archs:", torch.cuda.get_arch_list())
print("gpu:", torch.cuda.get_device_name(0))

cuda: 12.8
archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
gpu: Tesla P100-PCIE-16GB
